In [1]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
import warnings

warnings.filterwarnings("ignore")

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything
from rlearner_hill import get_rlearner_conversion

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data for Conversion...")
train_df = pd.read_csv(r"../../dataset/Hillstrom/Men/train_men.csv")
test_df =  pd.read_csv(r"../../dataset/Hillstrom/Men/test_men.csv")
val_df = pd.read_csv(r"../../dataset/Hillstrom/Men/val_men.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'conversion'
treatment_feature = 'treatment'

# Chuẩn bị X, y, treatment cho Train, Val, Test (Thêm .values để triệt tiêu UserWarning)
X_train = train_df[in_features].values
y_train = train_df[label_feature].values.astype(int) # Conversion là nhãn nhị phân (int)
t_train = train_df[treatment_feature].values.astype(int)

X_val = val_df[in_features].values
y_val_true = val_df[label_feature].values.flatten().astype(int)
t_val_true = val_df[treatment_feature].values.flatten().astype(int)

X_test = test_df[in_features].values
y_test_true = test_df[label_feature].values.flatten().astype(int)
t_test_true = test_df[treatment_feature].values.flatten().astype(int)

# Danh sách seeds
seeds = [412312, 42, 1874, 902745, 1]

seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective cho R-Learner Conversion
def objective(trial):
    # Không gian tham số tối ưu cho mô hình LightGBM (model_final của R-Learner)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }
    
    val_auqc_scores = []
    
    for SEED in seeds:
        # Khởi tạo R-Learner cho Conversion
        r_learner_model = get_rlearner_conversion(params, SEED)
        
        # Huấn luyện mô hình
        r_learner_model.fit(Y=y_train, T=t_train, X=X_train)
        
        # Dự đoán Uplift trên tập Val bằng hàm .effect()
        uplift_pred_val = r_learner_model.effect(X_val).flatten()
        
        # Đánh giá bằng AUQC
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (R-Learner Robust Conversion)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="R_Learner_Robust_Conv", sampler=fixed_sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")

print("Bộ tham số Robust tốt nhất:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# 4. Đánh giá bộ tham số tốt nhất trên tập TEST cho cả list seeds
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ R-LEARNER BEST PARAMS TRÊN TẬP TEST (CONVERSION)")
print("="*50)

test_results = []

for SEED in seeds:
    # Khởi tạo lại model với best params và seed hiện tại
    final_r_learner = get_rlearner_conversion(best_params, SEED)
    
    # Train
    final_r_learner.fit(Y=y_train, T=t_train, X=X_train)
    
    # Predict
    uplift_pred_test = final_r_learner.effect(X_test).flatten()
    
    # Đánh giá Metrics
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.3f}, AUQC={auqc_score:.3f}, Lift={lift_score:.3f}, KRCC={krcc_score:.3f}")

# Tính toán kết quả trung bình
results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()
std_results = results_df.std()

print("\n" + "*"*50)
print(f"🏆 KẾT QUẢ TRUNG BÌNH R-LEARNER CONVERSION ({len(seeds)} SEEDS) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f} ± {std_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f} ± {std_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f} ± {std_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f} ± {std_results['KRCC']:.3f}")

# Lưu kết quả
csv_filename = "r_learner_conversion_robust_results_men.csv"
results_df.to_csv(csv_filename, index=False)
print(f"\n💾 Đã lưu kết quả chi tiết từng seed vào: {csv_filename}")

/home/datnghiemxuan/hai/.venv_3_12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data for Conversion...
Locked random seed: 412312

🔃 Đang chạy Optuna Tuning (R-Learner Robust Conversion)...


Best trial: 25. Best value: 0.677113: 100%|██████████| 50/50 [36:28<00:00, 43.77s/it]


✅ Tuning hoàn tất! Best Mean Val AUQC: 0.6771
Bộ tham số Robust tốt nhất:
   n_estimators: 459
   learning_rate: 0.08825602130455591
   max_depth: 12
   num_leaves: 81
   min_child_samples: 181
   subsample: 0.6769575331922965
   colsample_bytree: 0.8671111145369625

🚀 ĐÁNH GIÁ R-LEARNER BEST PARAMS TRÊN TẬP TEST (CONVERSION)
Seed 412312: AUUC=0.514, AUQC=0.518, Lift=0.007, KRCC=0.008
Seed 42: AUUC=0.571, AUQC=0.576, Lift=0.009, KRCC=0.048
Seed 1874: AUUC=0.565, AUQC=0.571, Lift=0.009, KRCC=0.061
Seed 902745: AUUC=0.584, AUQC=0.589, Lift=0.009, KRCC=0.070
Seed 1: AUUC=0.607, AUQC=0.612, Lift=0.011, KRCC=0.030

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH R-LEARNER CONVERSION (5 SEEDS) 🏆
**************************************************
Mean AUUC: 0.568 ± 0.034
Mean AUQC: 0.573 ± 0.035
Mean Lift: 0.009 ± 0.001
Mean KRCC: 0.043 ± 0.025

💾 Đã lưu kết quả chi tiết từng seed vào: r_learner_conversion_robust_results_men.csv


# Women

In [2]:
import pandas as pd
import numpy as np
import optuna
from optuna.samplers import TPESampler
import warnings

warnings.filterwarnings("ignore")

import sys
sys.path.append("../..")
from metrics import auuc, auqc, lift, krcc
from utils import seed_everything
from rlearner_hill import get_rlearner_conversion

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Tải dữ liệu
print("Loading data for Conversion...")
train_df = pd.read_csv(r"../../dataset/Hillstrom/Women/train_women.csv")
test_df =  pd.read_csv(r"../../dataset/Hillstrom/Women/test_women.csv")
val_df = pd.read_csv(r"../../dataset/Hillstrom/Women/val_women.csv")

in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
               'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = 'conversion'
treatment_feature = 'treatment'

# Chuẩn bị X, y, treatment cho Train, Val, Test (Thêm .values để triệt tiêu UserWarning)
X_train = train_df[in_features].values
y_train = train_df[label_feature].values.astype(int) # Conversion là nhãn nhị phân (int)
t_train = train_df[treatment_feature].values.astype(int)

X_val = val_df[in_features].values
y_val_true = val_df[label_feature].values.flatten().astype(int)
t_val_true = val_df[treatment_feature].values.flatten().astype(int)

X_test = test_df[in_features].values
y_test_true = test_df[label_feature].values.flatten().astype(int)
t_test_true = test_df[treatment_feature].values.flatten().astype(int)

# Danh sách seeds
seeds = [412312, 42, 1874, 902745, 1]

seed_everything(seeds[0])

# 2. Định nghĩa hàm Objective cho R-Learner Conversion
def objective(trial):
    # Không gian tham số tối ưu cho mô hình LightGBM (model_final của R-Learner)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 10, 150),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 200),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }
    
    val_auqc_scores = []
    
    for SEED in seeds:
        # Khởi tạo R-Learner cho Conversion
        r_learner_model = get_rlearner_conversion(params, SEED)
        
        # Huấn luyện mô hình
        r_learner_model.fit(Y=y_train, T=t_train, X=X_train)
        
        # Dự đoán Uplift trên tập Val bằng hàm .effect()
        uplift_pred_val = r_learner_model.effect(X_val).flatten()
        
        # Đánh giá bằng AUQC
        score = auqc(y_val_true, t_val_true, uplift_pred_val, bins=100, plot=False)
        val_auqc_scores.append(score)
    
    return np.mean(val_auqc_scores)

# 3. Chạy Optuna Optimization
print("\n🔃 Đang chạy Optuna Tuning (R-Learner Robust Conversion)...")
fixed_sampler = TPESampler(seed=seeds[0])
study = optuna.create_study(direction="maximize", study_name="R_Learner_Robust_Conv", sampler=fixed_sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
print(f"✅ Tuning hoàn tất! Best Mean Val AUQC: {study.best_value:.4f}")

print("Bộ tham số Robust tốt nhất:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

# 4. Đánh giá bộ tham số tốt nhất trên tập TEST cho cả list seeds
print("\n" + "="*50)
print("🚀 ĐÁNH GIÁ R-LEARNER BEST PARAMS TRÊN TẬP TEST (CONVERSION)")
print("="*50)

test_results = []

for SEED in seeds:
    # Khởi tạo lại model với best params và seed hiện tại
    final_r_learner = get_rlearner_conversion(best_params, SEED)
    
    # Train
    final_r_learner.fit(Y=y_train, T=t_train, X=X_train)
    
    # Predict
    uplift_pred_test = final_r_learner.effect(X_test).flatten()
    
    # Đánh giá Metrics
    auuc_score = auuc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    auqc_score = auqc(y_test_true, t_test_true, uplift_pred_test, bins=100, plot=False)
    lift_score = lift(y_test_true, t_test_true, uplift_pred_test, h=0.3)
    krcc_score = krcc(y_test_true, t_test_true, uplift_pred_test, bins=100)
    
    test_results.append({
        'Seed': SEED,
        'AUUC': auuc_score,
        'AUQC': auqc_score,
        'Lift': lift_score,
        'KRCC': krcc_score
    })
    print(f"Seed {SEED}: AUUC={auuc_score:.3f}, AUQC={auqc_score:.3f}, Lift={lift_score:.3f}, KRCC={krcc_score:.3f}")

# Tính toán kết quả trung bình
results_df = pd.DataFrame(test_results)
mean_results = results_df.mean()
std_results = results_df.std()

print("\n" + "*"*50)
print(f"🏆 KẾT QUẢ TRUNG BÌNH R-LEARNER CONVERSION ({len(seeds)} SEEDS) 🏆")
print("*"*50)
print(f"Mean AUUC: {mean_results['AUUC']:.3f} ± {std_results['AUUC']:.3f}")
print(f"Mean AUQC: {mean_results['AUQC']:.3f} ± {std_results['AUQC']:.3f}")
print(f"Mean Lift: {mean_results['Lift']:.3f} ± {std_results['Lift']:.3f}")
print(f"Mean KRCC: {mean_results['KRCC']:.3f} ± {std_results['KRCC']:.3f}")

# Lưu kết quả
csv_filename = "r_learner_conversion_robust_results_women.csv"
results_df.to_csv(csv_filename, index=False)
print(f"\n💾 Đã lưu kết quả chi tiết từng seed vào: {csv_filename}")

Loading data for Conversion...
Locked random seed: 412312

🔃 Đang chạy Optuna Tuning (R-Learner Robust Conversion)...


Best trial: 4. Best value: 0.533527: 100%|██████████| 50/50 [14:16<00:00, 17.14s/it]


✅ Tuning hoàn tất! Best Mean Val AUQC: 0.5335
Bộ tham số Robust tốt nhất:
   n_estimators: 403
   learning_rate: 0.002414765171361614
   max_depth: 11
   num_leaves: 67
   min_child_samples: 42
   subsample: 0.6472333766858058
   colsample_bytree: 0.8082089562116921

🚀 ĐÁNH GIÁ R-LEARNER BEST PARAMS TRÊN TẬP TEST (CONVERSION)
Seed 412312: AUUC=0.690, AUQC=0.700, Lift=0.006, KRCC=0.078
Seed 42: AUUC=0.698, AUQC=0.705, Lift=0.006, KRCC=0.074
Seed 1874: AUUC=0.826, AUQC=0.838, Lift=0.007, KRCC=0.148
Seed 902745: AUUC=0.668, AUQC=0.677, Lift=0.006, KRCC=0.016
Seed 1: AUUC=0.716, AUQC=0.725, Lift=0.006, KRCC=0.093

**************************************************
🏆 KẾT QUẢ TRUNG BÌNH R-LEARNER CONVERSION (5 SEEDS) 🏆
**************************************************
Mean AUUC: 0.720 ± 0.062
Mean AUQC: 0.729 ± 0.063
Mean Lift: 0.006 ± 0.001
Mean KRCC: 0.082 ± 0.047

💾 Đã lưu kết quả chi tiết từng seed vào: r_learner_conversion_robust_results_women.csv
